# Hierarchical Stackelberg Security Problem

This is the authoritative implementation notebook. It follows the fixed 12-phase architecture and governing Phase 0 contracts. The Attacker best response is computed exclusively by the Bellman dynamic-programming solver; no CasADi/IPOPT NLP is part of the authoritative pipeline.

## Phase 1 — Project Initialization and Configuration

**Responsibility:** load the centralized configuration, create standard project paths, initialize logging, and validate configuration consistency.

This phase performs no terrain construction, LOS geometry, symbolic modeling, cost-map computation, optimization, export, or plotting.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()

if project_root.name == "p1b_4D":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from p1b_4D.configuration import build_configuration_bundle

configuration_bundle = build_configuration_bundle()

if not configuration_bundle["status"]["success"]:
    raise RuntimeError(configuration_bundle["status"]["message"])

configuration_bundle["validation"]["summary"]

2026-07-23 15:13:12,577 | INFO | stackelberg | phase=Phase 1: Configuration status=started


2026-07-23 15:13:12,578 | WARNING | stackelberg | phase=Phase 1: Configuration warning=Vehicle segment_length is deferred to the future transcription contract instead of fabricating a Phase 1 value


2026-07-23 15:13:12,579 | INFO | stackelberg | phase=Phase 1: Configuration status=success elapsed_seconds=0.000868


'Phase 1 configuration is valid'

## Phase 2 — Terrain Model

**Responsibility:** construct the authoritative terrain model, terrain derivatives and samples, terrain-following sensor position, fixed goal, sensor-dependent LOS tangent/boundary/masks, and LOS coverage area. Results are validated and exported without plotting.

In [2]:
from p1b_4D.geometry import build_geometry_bundle

phase_logger = configuration_bundle["primary_result"]["logging_utilities"]["logger"]
phase_context = configuration_bundle["primary_result"]["logging_utilities"]["phase_context"]
with phase_context(phase_logger, "Phase 2: Terrain and LOS Geometry") as phase_status:
    geometry_bundle = build_geometry_bundle(configuration_bundle)
    phase_status["warnings"].extend(geometry_bundle["status"]["warnings"])
    if not geometry_bundle["status"]["success"]:
        raise RuntimeError(geometry_bundle["status"]["message"])
geometry_bundle["validation"]["summary"]

2026-07-23 15:13:12,899 | INFO | stackelberg | phase=Phase 2: Terrain and LOS Geometry status=started


2026-07-23 15:13:12,907 | INFO | stackelberg | phase=Phase 2: Terrain and LOS Geometry status=success elapsed_seconds=0.007888


'Phase 2 geometry validation passed'

## Phase 3 — Sensor Geometry

Completed by the combined Phase 2 Geometry Bundle, which contains terrain-independent sensor placement, LOS tangent/boundary, masks, and coverage outputs.

## Phase 4 — CasADi Symbolic Detection Model

**Responsibility:** construct the authoritative CasADi symbolic range, LOS, powered acoustic, glide radar/radial-velocity/RCS, mission detection, mission time, normalization, and objective-component functions. Geometry is consumed from the Phase 2 Geometry Bundle and is not reconstructed.

In [3]:
from p1b_4D.detection import build_symbolic_detection_bundle

with phase_context(phase_logger, "Phase 3: CasADi Symbolic Detection") as phase_status:
    detection_bundle = build_symbolic_detection_bundle(
        configuration_bundle, geometry_bundle
    )
    phase_status["warnings"].extend(detection_bundle["status"]["warnings"])
    if not detection_bundle["status"]["success"]:
        raise RuntimeError(detection_bundle["status"]["message"])
detection_bundle["validation"]["summary"]

2026-07-23 15:13:12,923 | INFO | stackelberg | phase=Phase 3: CasADi Symbolic Detection status=started


2026-07-23 15:13:12,925 | INFO | stackelberg | phase=Phase 3: CasADi Symbolic Detection status=success elapsed_seconds=0.001986


'Phase 3 symbolic detection validation passed'

## Phase 5 — 4D Stage Cost Construction

**Responsibility:** construct the standard \(z,h,v,\gamma\) grids, state-validity masks, powered/glide detection and time components, normalized components, and the authoritative local glide \(J4D\). Invalid states receive positive-infinite cost. This phase performs no Bellman propagation or cost-to-go computation.

In [4]:
from p1b_4D.stage_cost import construct_stage_cost_4d

with phase_context(phase_logger, "Phase 4: 4D Stage Cost") as phase_status:
    stage_cost_4d_bundle = construct_stage_cost_4d(
        configuration_bundle, geometry_bundle, detection_bundle
    )
    phase_status["warnings"].extend(stage_cost_4d_bundle["status"]["warnings"])
    if not stage_cost_4d_bundle["status"]["success"]:
        raise RuntimeError(stage_cost_4d_bundle["status"]["message"])
stage_cost_4d_bundle["validation"]["summary"]

2026-07-23 15:13:12,931 | INFO | stackelberg | phase=Phase 4: 4D Stage Cost status=started


2026-07-23 15:13:16,998 | INFO | stackelberg | phase=Phase 4: 4D Stage Cost status=success elapsed_seconds=4.067717


'Phase 4 4D stage-cost validation passed'

## Phase 6 — 2D Projection

**Responsibility:** project the authoritative local \(J4D\) over feasible \(v,\gamma\) values and store diagnostic local controls. This result is visualization-only and is never a Bellman policy, value function, cost-to-go map, or trajectory source.

In [5]:
from p1b_4D.projection import construct_projected_cost_map

with phase_context(phase_logger, "Phase 5: 2D Projected Cost") as phase_status:
    projected_cost_bundle = construct_projected_cost_map(
        configuration_bundle,
        geometry_bundle,
        detection_bundle,
        stage_cost_4d_bundle,
    )
    phase_status["warnings"].extend(projected_cost_bundle["status"]["warnings"])
    if not projected_cost_bundle["status"]["success"]:
        raise RuntimeError(projected_cost_bundle["status"]["message"])
projected_cost_bundle["validation"]["summary"]

2026-07-23 15:13:17,008 | INFO | stackelberg | phase=Phase 5: 2D Projected Cost status=started


2026-07-23 15:13:17,016 | INFO | stackelberg | phase=Phase 5: 2D Projected Cost status=success elapsed_seconds=0.007743


'Phase 5 2D projected-cost validation passed'

## Phase 7 — Multi-start Bellman

**Responsibility:** run the multi-start coarse Bellman planner on authoritative `J4D`, preserving every start attempt and every feasible candidate. Bellman is the authoritative Attacker solver: the glide-phase dynamic-programming recursion is exact and converged for each exploration ordering and switching-point seed. Phase 8 selects the minimum-cost candidate directly as the Attacker best response. No filtering, continuous NLP refinement, Defender optimization, or plotting occurs here.

In [6]:
from p1b_4D.bellman import generate_bellman_candidates

with phase_context(phase_logger, "Phase 6: Multi-start Coarse Bellman") as phase_status:
    bellman_candidate_bundle = generate_bellman_candidates(
        configuration_bundle,
        geometry_bundle,
        detection_bundle,
        stage_cost_4d_bundle,
        projected_cost_bundle,
    )
    phase_status["warnings"].extend(bellman_candidate_bundle["status"]["warnings"])
    if not bellman_candidate_bundle["status"]["success"]:
        raise RuntimeError(bellman_candidate_bundle["status"]["message"])
bellman_candidate_bundle["validation"]["summary"]

2026-07-23 15:13:17,026 | INFO | stackelberg | phase=Phase 6: Multi-start Coarse Bellman status=started


2026-07-23 15:13:19,351 | WARNING | stackelberg | phase=Phase 6: Multi-start Coarse Bellman warning=6 switching starts did not produce candidates


2026-07-23 15:13:19,351 | INFO | stackelberg | phase=Phase 6: Multi-start Coarse Bellman status=success elapsed_seconds=2.324945


'Phase 6 multi-start Bellman candidate validation passed'

## Phase 8 — Bellman-Optimal Attacker Response

**Responsibility:** select the Bellman-optimal Attacker response directly from the multi-start Bellman candidate set via `select_authoritative_bellman_response`. The response is the minimum-mission-cost candidate produced by the exact, converged glide-phase dynamic-programming recursion over the discretized switching-point and state-action grid; the extracted cumulative stage cost is validated against the Bellman value at the initial state.

This is the sole authoritative Attacker best response. No CasADi/IPOPT NLP is used: `attacker_nlp.py` is retained only as a disconnected, deprecated experimental module for offline discretization-error comparison and is never called by this notebook. "Optimal" here means optimal on the discretized dynamic-programming formulation — not a claim of continuous global optimality.

In [7]:
from p1b_4D.bellman import select_authoritative_bellman_response

with phase_context(phase_logger, "Phase 7: Bellman Optimal Attacker Response") as phase_status:
    bellman_response_bundle = select_authoritative_bellman_response(
        bellman_candidate_bundle,
        configuration_bundle,
    )
    phase_status["warnings"].extend(bellman_response_bundle["status"]["warnings"])
    if not bellman_response_bundle["status"]["success"]:
        raise RuntimeError(bellman_response_bundle["status"]["message"])
bellman_response_bundle["validation"]["summary"]

2026-07-23 15:13:19,357 | INFO | stackelberg | phase=Phase 7: Bellman Optimal Attacker Response status=started


2026-07-23 15:13:19,358 | INFO | stackelberg | phase=Phase 7: Bellman Optimal Attacker Response status=success elapsed_seconds=0.000595


'Authoritative Bellman Attacker response validation passed'

## Phase 9 — Continuous Defender Optimization

**Responsibility:** expose a continuous `z_sensor` black-box evaluation and an algorithm-independent optimizer callback contract. Every callback evaluation rebuilds geometry and re-solves Bellman from scratch, selecting the Bellman-optimal Attacker response; sensor height remains `terrain(z_sensor) + mount_height`.

In [8]:
from p1b_4D.stackelberg_solver import (
    build_defender_optimizer_interface,
    evaluate_defender_position,
    solve_stackelberg_game,
)

with phase_context(phase_logger, "Phase 8: Continuous Defender Interface") as phase_status:
    defender_optimizer_interface = build_defender_optimizer_interface(
        configuration_bundle
    )
    phase_status["warnings"].extend(defender_optimizer_interface["status"]["warnings"])
    if not defender_optimizer_interface["status"]["success"]:
        raise RuntimeError(defender_optimizer_interface["status"]["message"])
    stackelberg_solution_bundle = solve_stackelberg_game(
        configuration_bundle
    )
    phase_status["warnings"].extend(stackelberg_solution_bundle["status"]["warnings"])
    if not stackelberg_solution_bundle["status"]["success"]:
        raise RuntimeError(stackelberg_solution_bundle["status"]["message"])
stackelberg_solution_bundle["validation"]["summary"]

2026-07-23 15:13:19,365 | INFO | stackelberg | phase=Phase 8: Continuous Defender Interface status=started


2026-07-23 15:15:37,073 | INFO | stackelberg | phase=Phase 8: Continuous Defender Interface status=success elapsed_seconds=137.707926


'Phase 9 Stackelberg solution validation passed'

## Phase 10 — Stackelberg Solver

`solve_stackelberg_game(configuration_bundle)` executes the default hierarchical coarse sweep, basin detection, and bounded Brent refinement. Every objective call performs a fresh complete nested Attacker solve through Bellman only.

## Phase 11 — Export

**Responsibility:** perform all active disk writes through the single standardized exporter. Every computational bundle (geometry, detection, stage cost, projected cost, Bellman candidates, the Bellman-optimal Attacker response, and the Stackelberg solution) is written as JSON metadata plus NPZ arrays.

In [9]:
from p1b_4D.result_export import export_all_results

with phase_context(phase_logger, "Phase 9: Standardized Result Export") as phase_status:
    result_export_status = export_all_results(
        configuration_bundle,
        geometry_bundle,
        detection_bundle,
        stage_cost_4d_bundle,
        projected_cost_bundle,
        bellman_candidate_bundle,
        bellman_response_bundle,
        stackelberg_solution_bundle,
    )
    phase_status["warnings"].extend(result_export_status["status"]["warnings"])
result_export_status["primary_result"]["export_status"]

2026-07-23 15:15:37,080 | INFO | stackelberg | phase=Phase 9: Standardized Result Export status=started


2026-07-23 15:15:39,303 | INFO | stackelberg | phase=Phase 9: Standardized Result Export status=success elapsed_seconds=2.222974


'complete'

## Phase 12 — Visualization

**Responsibility:** load standardized JSON/NPZ exports and render all five publication figures without calling geometry, cost, Bellman, or Defender computation modules.

In [10]:
from p1b_4D.result_import import import_result_collection
from p1b_4D.visualization import generate_project_visualizations

with phase_context(phase_logger, "Phase 10: Visualization") as phase_status:
    imported_result_collection = import_result_collection(
        result_export_status["primary_result"]["master_manifest_path"]
    )
    visualization_result = generate_project_visualizations(
        imported_result_collection,
        configuration_bundle["primary_result"]["project_paths"].figure_dir,
    )
    phase_status["warnings"].extend(visualization_result["status"]["warnings"])
visualization_result["primary_result"]["generated_figures"]

2026-07-23 15:15:39,542 | INFO | stackelberg | phase=Phase 10: Visualization status=started


2026-07-23 15:15:50,576 | INFO | stackelberg | phase=Phase 10: Visualization status=success elapsed_seconds=11.033991


('figure_1_geometry_overview',
 'figure_2_projected_cost',
 'figure_3_cost_to_go',
 'figure_4_all_paths',
 'figure_5_stackelberg_solution')

### Visualization implementation status

All five figures are generated exclusively from the complete standardized exported collection. The Attacker path shown in every figure is the Bellman-optimal discrete trajectory; no CasADi/IPOPT NLP output is plotted.